# Interface between Graphene and SiO2 (

## Introduction

This notebook demonstrates the creation of an interface between 2D and 3D material: Graphene on SiO2. 

Following the manuscript:
    > **Yong-Ju Kang, Joongoo Kang, and K. J. Chang** 
    > "Electronic structure of graphene and doping effect on SiO2"
    > Phys. Rev. B 78, 115404 (2008)
    > [DOI: 10.1103/PhysRevB.78.115404](https://doi.org/10.1103/PhysRevB.78.115404)



Builds the **metastable geometry of Sec. III** -- O-terminated α-quartz(0001) with the registry shifted so one surface O sits under a carbon atom and the other under a hexagon centre, and the back surface H-passivated. (Fig. 1(b) below shows the manuscript's *stable*, bonded registry for reference -- this notebook does not build that one.)

<img src='https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/interfaces/interface_2d_3d_graphene_silicon_dioxide/0-figure-from-manuscript.webp' width='600'/>


## 1. Prepare the Environment
### 1.1. Set up the notebook 

Set the following flags to control the notebook behavior 

In [ ]:
FILM_MILLER_INDICES = (0, 0, 1)
FILM_THICKNESS = 1  # in atomic layers
FILM_TERMINATION_FORMULA = None  # if None, the first termination will be used
FILM_VACUUM = 0.0  # in angstroms

SUBSTRATE_MILLER_INDICES = (0, 0, 1)
SUBSTRATE_THICKNESS = 7  # in atomic layers (for 14 bilayers -- from manuscript)
SUBSTRATE_TERMINATION_FORMULA = None  # if None, the first termination will be used
SUBSTRATE_VACUUM = 0.0  # in angstroms

INTERFACE_DISTANCE = 2.58  # Gap between substrate and film, in Angstrom -- from manuscript
INTERFACE_VACUUM = 20.0  # Vacuum over film, in Angstrom -- from manuscript

# Whether to convert materials to conventional cells before creating slabs.
USE_CONVENTIONAL_CELL = True

# Maximum area for the superlattice search algorithm (the final interface area will be smaller)
MAX_AREA = 150  # in Angstrom^2
# Additional fine-tuning parameters (increase values to get more strained matches):
MAX_AREA_TOLERANCE = 0.09  # in Angstrom^2
MAX_LENGTH_TOLERANCE = 0.05
MAX_ANGLE_TOLERANCE = 0.02

# Whether to reduce the resulting interface cell to the primitive cell after the interface creation.
REDUCE_RESULT_CELL_TO_PRIMITIVE = True

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Get input materials and assign `substrate` and `film`

In [ ]:
from mat3ra.standata.materials import Materials
from mat3ra.made.material import Material

film = Material.create(Materials.get_by_name_first_match("Graphene"))
substrate = Material.create(Materials.get_by_name_first_match("SiO2"))

### 1.4. Preview Substrate and Film

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([substrate, film], repetitions=[3, 3, 1], rotation="0x")

## 2. Configure slabs for interface

### 2.1. Get possible terminations for the slabs

In [ ]:
from mat3ra.made.tools.helpers import get_slab_terminations

film_slab_terminations = get_slab_terminations(material=film, miller_indices=FILM_MILLER_INDICES)
substrate_slab_terminations = get_slab_terminations(material=substrate, miller_indices=SUBSTRATE_MILLER_INDICES)
print("Film slab terminations:", film_slab_terminations)
print("Substrate slab terminations:", substrate_slab_terminations)


### 2.2. Visualize slabs for all possible terminations

In [ ]:
from mat3ra.made.tools.helpers import create_slab, select_slab_termination

film_slabs = [create_slab(film, miller_indices=FILM_MILLER_INDICES, termination_top=termination, vacuum=0) for termination
              in
              film_slab_terminations]

substrate_slabs = [create_slab(substrate, miller_indices=SUBSTRATE_MILLER_INDICES, termination_top=termination, vacuum=0)
                   for termination in
                   substrate_slab_terminations]

film_slabs_with_titles = [{"material": slab, "title": str(termination)} for slab, termination in
                          zip(film_slabs, film_slab_terminations)]
substrate_slabs_with_titles = [{"material": slab, "title": str(termination)} for slab, termination in
                               zip(substrate_slabs, substrate_slab_terminations)]

visualize(film_slabs_with_titles, repetitions=[3, 3, 1], rotation="-90x")
visualize(substrate_slabs_with_titles, repetitions=[3, 3, 1], rotation="-90x")


### 2.3. Select terminations for the Slabs

In [ ]:
film_termination = select_slab_termination(film_slab_terminations, FILM_TERMINATION_FORMULA)
substrate_termination = select_slab_termination(substrate_slab_terminations, SUBSTRATE_TERMINATION_FORMULA)

### 2.4. Create Substrate and Film Slabs
Slab Configuration lets define the slab thickness, vacuum, and the Miller indices of the interfacial plane.
Define the substrate slab cell that will be used as a base for the interface and the film slab cell that will be placed on top of the substrate slab.

In [ ]:

from mat3ra.made.tools.build.pristine_structures.two_dimensional.slab import SlabConfiguration, SlabBuilder

substrate_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=substrate,
    miller_indices=SUBSTRATE_MILLER_INDICES,
    number_of_layers=SUBSTRATE_THICKNESS,
    vacuum=0.0,
    termination_top_formula=SUBSTRATE_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL
)

film_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=film,
    miller_indices=FILM_MILLER_INDICES,
    number_of_layers=FILM_THICKNESS,
    vacuum=0.0,
    termination_bottom_formula=FILM_TERMINATION_FORMULA, # film termination is on the bottom
    use_conventional_cell=USE_CONVENTIONAL_CELL
)

substrate_slab = SlabBuilder().get_material(substrate_slab_config)
film_slab = SlabBuilder().get_material(film_slab_config)

## 3. Analyze possible interfaces with ZSL Analyzer
### 3.1. Initialize ZSL Analyzer
The search algorithm for supercells matching can be tuned by setting its parameters directly, otherwise the default values are used.


In [ ]:
from mat3ra.made.tools.analyze.interface import ZSLInterfaceAnalyzer

zsl_analyzer = ZSLInterfaceAnalyzer(
    substrate_slab_configuration=substrate_slab_config,
    film_slab_configuration=film_slab_config,
    max_area=MAX_AREA,
    max_area_ratio_tol=MAX_AREA_TOLERANCE,
    max_length_tol=MAX_LENGTH_TOLERANCE,
    max_angle_tol=MAX_ANGLE_TOLERANCE,
    reduce_result_cell=False  # Reduces supercell matrices in analyzer
)


### 3.2. Generate matches with strain analyzer
Matches are sorted by size and strain.


In [ ]:
matches = zsl_analyzer.zsl_match_holders


### 3.3. Plot matches by area and strain


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.plot import plot_strain_vs_area

PLOT_SETTINGS = {
    "HEIGHT": 600,
    "X_SCALE": "log",  # or linear
    "Y_SCALE": "log",  # or linear
}

plot_strain_vs_area(matches, PLOT_SETTINGS)

### 3.4. Select the interface

Select the index for the interface with the lowest strain and the smallest area.


In [ ]:
selected_index = 0

from mat3ra.made.tools.helpers import create_interface_zsl_between_slabs as create_zsl_interface_between_slabs

interface = create_zsl_interface_between_slabs(
    substrate_slab=substrate_slab,
    film_slab=film_slab,
    gap=INTERFACE_DISTANCE,
    vacuum=INTERFACE_VACUUM,
    match_id=selected_index,
    max_area=MAX_AREA,
    max_area_ratio_tol=MAX_AREA_TOLERANCE,
    max_length_tol=MAX_LENGTH_TOLERANCE,
    max_angle_tol=MAX_ANGLE_TOLERANCE,
    reduce_result_cell_to_primitive=REDUCE_RESULT_CELL_TO_PRIMITIVE,
)


## 4. Preview the selected interface


In [ ]:
visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")


## 5. Correct the geometry to the paper's metastable registry

As built, the ZSL interface has a 60° cell (made's symbolic K-point assumes 120°) and a registry close to the manuscript's *stable* bonded geometry: both surface O atoms sit near a C atom. Sec. III's metastable geometry needs one O under a C and the other under a hexagon centre, and the manuscript H-passivates the back surface; neither follows from `create_interface_zsl_between_slabs` alone.

### 5.1. Save the strained free-standing film as the Dirac-point reference

Taken off the film part of the as-built interface -- same strain as in the interface, no substrate -- before the cell reset, registry shift and passivation below, which only act on the interface.

In [ ]:
from mat3ra.made.tools.helpers import create_supercell
from mat3ra.made.tools.modify import interface_get_part, add_vacuum
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum

film_reference = interface_get_part(interface, InterfacePartsEnum.FILM)
film_reference = create_supercell(film_reference, supercell_matrix=[[1, 0, 0], [-1, 1, 0], [0, 0, 1]])
film_reference.lattice.type = "HEX"
film_reference = add_vacuum(film_reference, vacuum=INTERFACE_VACUUM, on_top=True, to_bottom=True)
film_reference.name = "Graphene 2x2 on SiO2, free-standing reference"
print(f"{film_reference.basis.number_of_atoms} atoms, gamma = {film_reference.lattice.gamma:.3f} deg")

### 5.2. Reset the interface cell to a 120° hexagonal setting

In [ ]:
# The ZSL match above is a 60° cell; made's symbolic K point assumes 120°, so the
# hexagonal path point is meaningless until the cell is re-set (same physics, same atoms).
interface = create_supercell(interface, supercell_matrix=[[1, 0, 0], [-1, 1, 0], [0, 0, 1]])
interface.lattice.type = "HEX"
print(f"gamma = {interface.lattice.gamma:.3f} deg, a = {interface.lattice.a:.4f} Å")

### 5.3. Shift the film to the manuscript's registry

`REGISTRY_SHIFT` is the balanced choice: the real lattice cannot put one surface O exactly under a C while the other sits exactly over a hexagon centre (a 0.71 Å misfit between the two O positions and the graphene lattice), so the shift splits that misfit evenly, leaving each O 0.35 Å from its ideal site rather than putting one exactly on site and the other off by the full 0.71 Å.

In [ ]:
import numpy as np
from collections import Counter
from mat3ra.made.tools.modify import interface_displace_part

REGISTRY_SHIFT = [-1.010994, -0.724791, 0.0]  # Angstrom, cartesian, applied to the film

interface = interface_displace_part(interface, REGISTRY_SHIFT, label=InterfacePartsEnum.FILM)


def cartesian_coordinates(material):
    clone = material.clone()
    clone.to_cartesian()
    return np.array(clone.basis.coordinates.values)


def xy_distance(p, q, cell_xy):
    """Minimum-image in-plane distance; the two lattice vectors are periodic, z is not."""
    d_xy = (p - q)[:2]
    return min(np.linalg.norm(d_xy - i * cell_xy[0] - j * cell_xy[1])
               for i in (-1, 0, 1) for j in (-1, 0, 1))


elements = interface.basis.elements.values
coordinates = cartesian_coordinates(interface)
cell_xy = np.array(interface.lattice.vector_arrays)[:2, :2]
carbon_indices = [i for i, el in enumerate(elements) if el == "C"]
oxygen_indices = [i for i, el in enumerate(elements) if el == "O"]
surface_oxygens = sorted(oxygen_indices, key=lambda i: -coordinates[i, 2])[:2]
for i in surface_oxygens:
    nearest_c = min(xy_distance(coordinates[i], coordinates[j], cell_xy) for j in carbon_indices)
    print(f"O at z={coordinates[i, 2]:.4f}: nearest C at {nearest_c:.4f} Å")

### 5.4. Passivate the back surface

`passivate_surface` places one H per exposed atom. The back Si is missing two bonds (coordination 2 of 4), so one dangling bond remains after this call -- declared, not chased further; see the tutorial's Troubleshooting section.

In [ ]:
from mat3ra.made.tools.helpers import passivate_surface, SurfaceTypesEnum

GBRV_VALENCE = {"Si": 4, "O": 6, "C": 4, "H": 1}

natoms_before = interface.basis.number_of_atoms
interface = passivate_surface(interface, passivant="H", surface=SurfaceTypesEnum.BOTTOM)
interface.basis.set_labels_from_list([])
composition = Counter(interface.basis.elements.values)
electron_count = sum(GBRV_VALENCE[element] * count for element, count in composition.items())
print(f"H added = {interface.basis.number_of_atoms - natoms_before}; "
      f"{interface.basis.number_of_atoms} atoms, composition {dict(composition)}")
print(f"{electron_count} valence electrons ({'odd' if electron_count % 2 else 'even'}, GBRV: "
      "Si 4, O 6, C 4, H 1)")
interface.name = "Graphene 2x2 on SiO2, metastable registry, H-passivated"

### 5.5. Preview the corrected interface

In [ ]:
visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

## 6. Pass data to the outside runtime


In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials([interface, film_reference])
download_content_to_file(interface.to_json(), f"{interface.name}.json")